In [1]:
import pandas as pd
import re
import string
from matplotlib import pyplot as plt
import re
import string
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [2]:
train = pd.read_csv("./DataSets/train (2).csv")
test = pd.read_csv("./DataSets/test (1).csv")
val = pd.read_csv("./DataSets/dev.csv")
sexism_data = pd.read_csv("./DataSets/sexism_data.csv", encoding="latin1")
classified_test = pd.read_csv("./DataSets/classified_tweets.csv")
twitter_sexism = pd.read_csv("./DataSets/twitter_sexism_parsed_dataset.csv")

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /home/yogesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/yogesh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/yogesh/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
train.drop(columns=['rewire_id', 'label_category', 'label_vector', 'split'], inplace=True)
test.drop(columns=['rewire_id', 'label_category', 'label_vector', 'split'], inplace=True)
val.drop(columns=['rewire_id', 'label_category', 'label_vector', 'split'], inplace=True)

In [5]:
def change_column_names(df, text_col, label_col):
    return df.rename(columns={ text_col: "text", label_col: "target"})

In [6]:
train = change_column_names(train, "text", "label_sexist")
test = change_column_names(test, "text", "label_sexist")
val = change_column_names(val, "text", "label_sexist")

In [7]:
train.head()

,text,target
0,"Then, she's a keeper. 😉",not sexist
1,This is like the Metallica video where the poo...,not sexist
2,woman?,not sexist
3,Unlicensed day care worker reportedly tells co...,not sexist
4,[USER] Leg day is easy. Hot girls who wear min...,sexist


In [8]:
stop_words = set(stopwords.words("english"))
negations = {"no", "not", "nor", "never"}
stop_words = stop_words - negations

In [9]:
lemmatizer = WordNetLemmatizer()

In [10]:
def clean_text(text: str) -> str:
    # lowercase
    text = text.lower()

    # convert emojis to text
    # 😂 -> face_with_tears_of_joy
    text = emoji.demojize(text, delimiters=(" ", " "))

    # replace urls, users, numbers
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\d+", " NUM ", text)

    # normalize repeated punctuation
    text = re.sub(r"!{2,}", " EXCLAMATION ", text)
    text = re.sub(r"\?{2,}", " QUESTION ", text)

    # remove punctuation (underscores are kept for emoji words)
    punctuation = string.punctuation.replace("_", "")
    text = text.translate(str.maketrans("", "", punctuation))

    # tokenize
    tokens = text.split()

    # remove stopwords + lemmatize
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(tokens)


In [11]:
print(train['target'].value_counts())
print(test['target'].value_counts())
print(val['target'].value_counts())

target
not sexist    10602
sexist         3398
Name: count, dtype: int64
target
not sexist    3030
sexist         970
Name: count, dtype: int64
target
not sexist    1514
sexist         486
Name: count, dtype: int64


In [12]:
train["clean_text"] = train["text"].apply(clean_text)
test["clean_text"] = test["text"].apply(clean_text)
val["clean_text"] = val["text"].apply(clean_text)

In [13]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)

In [14]:
train.columns

Index(['text', 'target', 'clean_text'], dtype='object')

In [15]:
label_map = {
    "not sexist": 0,
    "sexist": 1
}

In [16]:
train["target"] = train["target"].str.strip().str.lower().map(label_map)
test["target"] = test["target"].str.strip().str.lower().map(label_map)
val["target"] = val["target"].str.strip().str.lower().map(label_map)

In [17]:
x_train = train['clean_text'].values
x_test = test['clean_text'].values
x_val = val['clean_text'].values

y_train = train['target'].values
y_test = test['target'].values
y_val = val['target'].values

In [18]:
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)
x_val_vec = vectorizer.transform(x_val)

In [19]:
logReg = LogisticRegression(max_iter=1000, class_weight="balanced")
svc = svc = LinearSVC(class_weight="balanced")
rfc = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=42)

In [20]:
logReg.fit(x_train_vec, y_train)
svc.fit(x_train_vec, y_train)
rfc.fit(x_train_vec, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [21]:
logReg_pred = logReg.predict(x_test_vec)
svc_pred = svc.predict(x_test_vec)
rfc_pred = rfc.predict(x_test_vec)

In [22]:
print(classification_report(y_true=y_test, y_pred=logReg_pred))
print(classification_report(y_true=y_test, y_pred=svc_pred))
print(classification_report(y_true=y_test, y_pred=rfc_pred))

              precision    recall  f1-score   support

           0       0.89      0.84      0.87      3030
           1       0.58      0.69      0.63       970

    accuracy                           0.80      4000
   macro avg       0.74      0.76      0.75      4000
weighted avg       0.82      0.80      0.81      4000

              precision    recall  f1-score   support

           0       0.88      0.86      0.87      3030
           1       0.60      0.63      0.61       970

    accuracy                           0.81      4000
   macro avg       0.74      0.75      0.74      4000
weighted avg       0.81      0.81      0.81      4000

              precision    recall  f1-score   support

           0       0.88      0.90      0.89      3030
           1       0.67      0.62      0.64       970

    accuracy                           0.83      4000
   macro avg       0.78      0.76      0.77      4000
weighted avg       0.83      0.83      0.83      4000

